# Montgomery + Shenzhen mask-emission verify

End-to-end check that `output_mask=True` produces real, sensible masks for both NLM TB chest-X-ray datasets after the mask-fusion wiring was added.

**What this notebook verifies:**
1. The `mask_output_dir` preprocess step writes one PNG per image (idempotent).
2. The harmonized DataFrame ends up with a populated absolute `mask_path` column.
3. Samples emitted from `get_datasets()` include a `mask` tensor of the same shape as `img`.
4. Visually overlaid masks look like real lung outlines (Montgomery) and real TB-region blobs (Shenzhen positives) — black masks for Shenzhen negatives.

**Conventions** (after the recent refactor):
- `base_image_dir` for both datasets is now the `CXR_png/` directory itself, not the parent.
- Montgomery fuses left+right manual lung masks → single binary lung mask.
- Shenzhen unions per-finding region masks → single binary TB-region mask; TB-negative images get a same-size all-zero PNG (matches SIIM-ACR PTX convention).

Run top-to-bottom. First run writes the fused PNGs (~25 s for Montgomery, ~60 s for Shenzhen). Subsequent runs are no-ops thanks to the `os.path.isfile` skip.

## Section 0 — Setup

Edit the four paths in the next cell to match your environment. Defaults are placeholders — point them at your local copy.

In [ ]:
from pathlib import Path

# ── inputs ────────────────────────────────────────────────────────────
MONT_BASE_IMG_DIR = "/path/to/Montgomery-CXR/MontgomerySet/CXR_png/"
SHEN_BASE_IMG_DIR = "/path/to/Shenzhen-Hospital-CXR-Set/CXR_png/"

# ── outputs (where the fused mask PNGs go) ────────────────────────────
# Use a local scratch dir so we don't write back into NAS.
MASK_OUTPUT_ROOT = Path("/path/to/mask_verify_out/").resolve()
MONT_MASK_DIR = MASK_OUTPUT_ROOT / "montgomery_fused"
SHEN_MASK_DIR = MASK_OUTPUT_ROOT / "shenzhen_tb_region"
MASK_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"mask output root: {MASK_OUTPUT_ROOT}")

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image

from radharmony.dataset import MontgomeryCXRDataset, ShenzhenCXRDataset

# Tensor → numpy helper that survives bf16 (matplotlib can't handle bf16)
def to_np(t):
    return t.detach().float().cpu().numpy() if torch.is_tensor(t) else np.asarray(t)

def show_overlay(ax, img_t, mask_t, title="", mask_color=(1.0, 0.0, 0.0), alpha=0.35):
    """Overlay a binary/continuous mask in red on top of a grayscale image."""
    img = to_np(img_t)
    if img.ndim == 3:        # (C, H, W) → take first channel
        img = img[0]
    img = (img - img.min()) / max(img.max() - img.min(), 1e-8)
    ax.imshow(img, cmap="gray", interpolation="nearest")
    mask = to_np(mask_t)
    if mask.ndim == 3:
        mask = mask[0]
    mask = (mask > mask.max() * 0.5).astype(np.float32)  # binarize
    overlay = np.zeros((*mask.shape, 4), dtype=np.float32)
    overlay[..., 0] = mask_color[0]
    overlay[..., 1] = mask_color[1]
    overlay[..., 2] = mask_color[2]
    overlay[..., 3] = mask * alpha
    ax.imshow(overlay, interpolation="nearest")
    ax.set_title(title, fontsize=10)
    ax.axis("off")

---
## Section 1 — Montgomery-CXR

138 PA chest radiographs (80 normal, 58 TB-positive). Each image has hand-drawn left and right lung masks under sibling `ManualMask/{leftMask,rightMask}/`. `preprocess_masks()` ORs them into a single binary lung mask per image.

In [ ]:
mont_ds = MontgomeryCXRDataset(
    base_image_dir=MONT_BASE_IMG_DIR,
    cache_dir=None,                          # disable PersistentDataset cache for this verify
    mask_output_dir=str(MONT_MASK_DIR),
    output_cls=True,
    output_mask=True,
)
df_mont = mont_ds.get_harmonized_df()
print(f"rows: {len(df_mont)}")
print(f"columns: {list(df_mont.columns)}")
print(f"label distribution (tb):\n{df_mont['tb'].value_counts(dropna=False).to_string()}")
df_mont.head(3)

In [ ]:
# Sanity check: every row has an absolute mask_path that exists on disk
missing = [p for p in df_mont["mask_path"] if not (isinstance(p, str) and os.path.isfile(p))]
n_fused = len(list(MONT_MASK_DIR.glob("*.png")))
print(f"PNGs in {MONT_MASK_DIR}: {n_fused}")
print(f"rows with missing/invalid mask_path: {len(missing)}  (expect 0)")
print(f"mask_path is absolute: {all(p.startswith('/') for p in df_mont['mask_path'] if isinstance(p, str))}")

In [ ]:
# Pull samples through the MONAI dataset pipeline + visualize
torch_ds = mont_ds.get_datasets(n_splits=None, num_cores=1)
print(f"torch dataset length: {len(torch_ds)}  (expect {len(df_mont)})")

# Pick 2 normal + 2 TB samples
normal_indices = df_mont.index[df_mont["tb"] == 0].tolist()[:2]
tb_indices     = df_mont.index[df_mont["tb"] == 1].tolist()[:2]
chosen = normal_indices + tb_indices

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, idx in zip(axes, chosen):
    s = torch_ds[idx]
    label = "TB" if float(s["cls"].item()) == 1.0 else "normal"
    pos_frac = float((s["mask"] > 0).float().mean())
    show_overlay(
        ax, s["img"], s["mask"],
        title=f"idx={idx}  cls={label}\nmask positive frac={pos_frac:.3f}",
    )
fig.suptitle("Montgomery — fused left+right lung mask (red overlay)", y=1.02)
plt.tight_layout()
plt.show()

**Expected for Montgomery:** Every image (normal or TB) should have a roughly lung-shaped red overlay covering ~25–45 % of the frame. The mask is just the anatomy outline — it does **not** localize the TB; the `tb` label does that.

---
## Section 2 — Shenzhen-Hospital-CXR-Set

662 frontal CXRs (336 TB-positive, 326 normal). `Annotations-2/masks/` contains per-finding region masks (e.g. `CHNCXR_0327_1_Pleural_Effusion_2.png`) only for TB-positive cases. `preprocess_masks()` unions them per image; TB-negatives get a same-size all-zero PNG (SIIM convention).

In [ ]:
shen_ds = ShenzhenCXRDataset(
    base_image_dir=SHEN_BASE_IMG_DIR,
    cache_dir=None,
    mask_output_dir=str(SHEN_MASK_DIR),
    output_cls=True,
    output_mask=True,
)
df_shen = shen_ds.get_harmonized_df()
print(f"rows: {len(df_shen)}")
print(f"label distribution (tuberculosis):\n{df_shen['tuberculosis'].value_counts(dropna=False).to_string()}")

# Sanity check: every row has a mask_path that exists
missing = [p for p in df_shen["mask_path"] if not (isinstance(p, str) and os.path.isfile(p))]
n_fused = len(list(SHEN_MASK_DIR.glob("*.png")))
print(f"\nPNGs in {SHEN_MASK_DIR}: {n_fused}")
print(f"rows with missing/invalid mask_path: {len(missing)}  (expect 0)")

In [ ]:
# Confirm TB-negatives get a black PNG, TB-positives get a non-trivial one — at the PNG level
import json

def mask_positive_frac(path):
    arr = np.asarray(Image.open(path).convert("L"))
    return float((arr > 0).mean())

df_check = df_shen[["image_path", "tuberculosis", "mask_path"]].copy()
df_check["mask_pos_frac"] = df_check["mask_path"].apply(mask_positive_frac)
df_check["finding_masks_count"] = df_shen["finding_masks"].apply(
    lambda x: len(json.loads(x)) if isinstance(x, str) else 0
)

print("By TB label — mask positive fraction (should be 0 for tuberculosis=0):")
print(df_check.groupby("tuberculosis")["mask_pos_frac"].describe().round(4))
print("\nBy TB label — finding_masks list length:")
print(df_check.groupby("tuberculosis")["finding_masks_count"].describe().round(2))

In [ ]:
# Visualize 2 TB-positive + 2 TB-negative samples
torch_ds = shen_ds.get_datasets(n_splits=None, num_cores=1)
print(f"torch dataset length: {len(torch_ds)}  (expect {len(df_shen)})")

# Pick rows with the highest mask positive fraction (most visible TB regions) +
# any two negatives for sanity.
ranked_pos = df_check[df_check["tuberculosis"] == 1].sort_values(
    "mask_pos_frac", ascending=False
).index.tolist()[20:22]
any_neg = df_check[df_check["tuberculosis"] == 0].index.tolist()[:2]
chosen = ranked_pos + any_neg

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, idx in zip(axes, chosen):
    s = torch_ds[idx]
    label = "TB" if float(s["cls"].item()) == 1.0 else "normal"
    pos_frac = float((s["mask"] > 0).float().mean())
    n_findings = df_check.loc[idx, "finding_masks_count"]
    show_overlay(
        ax, s["img"], s["mask"],
        title=f"idx={idx}  cls={label}  n_findings={n_findings}\nmask positive frac={pos_frac:.4f}",
    )
    
fig.suptitle("Shenzhen — unioned per-finding TB-region mask (red overlay)", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
plt.imshow(s["mask"][0].float().numpy())

**Expected for Shenzhen:**
- TB-positive panels: small red blobs at the actual finding locations (cavity / infiltrate / effusion); mask positive frac typically 0.001–0.05.
- TB-negative panels: clean grayscale CXR with **no red overlay** (black mask, positive frac = 0).

---
## Section 3 — Cache + reload check

Rebuild both datasets without deleting the mask output dirs. The `Fusing …` progress bars should be very fast — every PNG is `os.path.isfile`-skipped.

In [ ]:
import time

for name, cls, base, mask_dir in [
    ("Montgomery", MontgomeryCXRDataset, MONT_BASE_IMG_DIR, str(MONT_MASK_DIR)),
    ("Shenzhen",   ShenzhenCXRDataset,   SHEN_BASE_IMG_DIR, str(SHEN_MASK_DIR)),
]:
    t0 = time.time()
    ds = cls(
        base_image_dir=base,
        cache_dir=None,
        mask_output_dir=mask_dir,
        output_cls=True,
        output_mask=True,
    )
    df = ds.get_harmonized_df()
    dt = time.time() - t0
    n_pngs = len(list(Path(mask_dir).glob("*.png")))
    print(f"{name}: harmonize + reuse fused PNGs in {dt:.2f}s  (rows={len(df)}, pngs on disk={n_pngs})")

---
## Section 4 — Summary checklist

If everything above renders correctly, all of these should be true:

- [ ] Montgomery emits 138 samples, every one has a non-trivial lung mask (positive frac > 0).
- [ ] Shenzhen emits 662 samples; TB-positive masks have positive frac > 0, TB-negative masks have positive frac = 0.
- [ ] `mask_path` is absolute and points at the user-chosen `mask_output_dir` (not the NAS, not the source `ManualMask/` / `Annotations-2/`).
- [ ] Section 3 finishes in well under a second per dataset (idempotency works).
- [ ] Visual overlays look anatomically correct.